# 📚 [통계적 검정과 인과추론] 종합 과제

- ## 📊랜딩 페이지 A/B 테스트 
📌 **가설 검정 → 효과 크기 → 인과 추론**


## 1. 배경 및 질문
- 한 이커머스 회사가 **새 랜딩 페이지(new_page)** 를 만들었다.
- 핵심 질문: **새 페이지가 기존 페이지(old_page)보다 전환율을 높이는가? 그렇다면 도입할 가치가 있는가?**


### 1-1. 데이터 소개
**데이터 개요**
| 항목 | 내용 |
|---|---|
| 출처 | Kaggle — [랜딩 페이지 A/B 테스트 데이터셋](https://www.kaggle.com/datasets/zhangluyuan/ab-testing) |
| 내려받은 날짜 | 2026-11-06 |
| 규모 | 294,478행 × 5열 |
| 기간 | 2017-01-02 ~ 2017-01-24 (약 3주) |
| 단위 | 사용자 1명의 방문 기록 |   
     
**컬럼 설명**
| 컬럼 | 의미 |
|---|---|
| user_id | 사용자 ID |
| timestamp | 방문 시각 |
| group | 배정 그룹 (control = 대조군 / treatment = 실험군) |
| landing_page | 실제로 본 페이지 (old_page / new_page) |
| converted | 전환 여부 (1 = 전환, 0 = 미전환) |

-  이 분석은 난수를 사용하지 않으므로 별도의 시드 고정이 필요 없다.

In [1]:
# ── 라이브러리 ──
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import (proportions_ztest,
                                          confint_proportions_2indep,
                                          proportion_effectsize)
from statsmodels.stats.power import NormalIndPower

# ── 데이터 불러오기 ──────────────────────
df = pd.read_csv('랜딩 페이지 테스트.csv')
df.shape

(294478, 5)

In [2]:
import pandas as pd

df = pd.read_csv('랜딩 페이지 테스트.csv')
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   user_id       294478 non-null  int64
 1   timestamp     294478 non-null  str  
 2   group         294478 non-null  str  
 3   landing_page  294478 non-null  str  
 4   converted     294478 non-null  int64
dtypes: int64(2), str(3)
memory usage: 23.0 MB


## 2. 데이터 점검

### 2-1. 중복 확인하기

| 단계 | 확인 내용 | 결과 |
|---|---|---|
| a | 모든 값이 똑같은 행이 있는가? | 0건 |
| b | 같은 user_id가 두 번 이상 나오는가? | 3,894건 (고유 사용자 290,584명) |
| c | 중복된 사용자는 어떻게 생겼는가? | 직접 확인 |
| d | 중복에는 어떤 유형이 있는가? | 3가지 유형 |

In [4]:
#같은 행이 있는지 확인하기
df.duplicated().sum()

np.int64(0)

In [5]:
#같은 user_id가 있는지 확인하기
print(df['user_id'].duplicated().sum())   # 중복 등장 횟수
print(df['user_id'].nunique())            # 고유 사용자 수

3894
290584


In [6]:
#중복된 사용자 직접 확인하기
dup = df[df['user_id'].duplicated(keep=False)].sort_values('user_id')
dup.head(6)


,user_id,timestamp,group,landing_page,converted
230259,630052,2017-01-17 01:16:05.208766,treatment,new_page,0
213114,630052,2017-01-07 12:25:54.089486,treatment,old_page,1
22513,630126,2017-01-14 13:35:54.778695,treatment,old_page,0
251762,630126,2017-01-19 17:16:00.280440,treatment,new_page,0
183371,630137,2017-01-20 02:08:49.893878,control,old_page,0
11792,630137,2017-01-22 14:59:22.051308,control,new_page,0


#### d. 중복 유형 분류하기

| 그룹 수 | 페이지 수 | 의미 |
|---|---|---|
| 1 | 2 | 그룹은 같은데 페이지가 다름 → 두 페이지를 모두 봄 |
| 2 | 1 | 페이지는 같은데 그룹 라벨이 다름 → 라벨이 바뀜 |
| 1 | 1 | 그룹·페이지 모두 같음 → 같은 기록이 2번 |

In [7]:
# 1) 사용자별로 그룹 종류 수, 페이지 종류 수 세기
dup_type = dup.groupby('user_id').agg(
    그룹수=('group', 'nunique'),
    페이지수=('landing_page', 'nunique')
)

# 2) 조합에 이름 붙이기
def label(row):
    if row['그룹수'] == 1 and row['페이지수'] == 2:
        return '그룹 같음 · 페이지 다름'
    elif row['그룹수'] == 2 and row['페이지수'] == 1:
        return '페이지 같음 · 그룹 다름'
    else:
        return '완전히 같은 기록'

dup_type['유형'] = dup_type.apply(label, axis=1)

# 3) 유형별 인원 세기
dup_type['유형'].value_counts()

유형
그룹 같음 · 페이지 다름    1998
페이지 같음 · 그룹 다름    1895
완전히 같은 기록            1
Name: count, dtype: int64

In [8]:
for t in dup_type['유형'].unique():
    uid = dup_type[dup_type['유형'] == t].index[0]   # 유형별 첫 번째 사용자
    print(f'▶ {t} (user_id {uid})')
    display(df[df['user_id'] == uid])

▶ 그룹 같음 · 페이지 다름 (user_id 630052)


,user_id,timestamp,group,landing_page,converted
213114,630052,2017-01-07 12:25:54.089486,treatment,old_page,1
230259,630052,2017-01-17 01:16:05.208766,treatment,new_page,0


▶ 페이지 같음 · 그룹 다름 (user_id 630320)


,user_id,timestamp,group,landing_page,converted
207211,630320,2017-01-07 18:02:43.626318,control,old_page,0
255753,630320,2017-01-12 05:27:37.181803,treatment,old_page,0


▶ 완전히 같은 기록 (user_id 773192)


,user_id,timestamp,group,landing_page,converted
1899,773192,2017-01-09 05:37:58.781806,treatment,new_page,0
2893,773192,2017-01-14 02:55:59.590927,treatment,new_page,0


#### 📝 2-1 정리
- 완전히 같은 행: 0건
- user_id 중복: 3,894건 (고유 사용자 290,584명)
- 중복 유형
  | 유형 | 인원 | 의미 |
  |---|---|---|
  | 그룹 같음 · 페이지 다름 | 1,998명 | 배정된 그룹과 다른 페이지가 한 번 노출(또는 기록)됨 |
  | 페이지 같음 · 그룹 다름 | 1,895명 | 같은 페이지를 봤는데 그룹 라벨이 바뀌어 기록됨 |
  | 완전히 같은 기록 | 1명 | 같은 기록이 2번 저장됨 |
- 앞의 두 유형은 모두 **배정된 그룹과 본 페이지가 어긋난 행**을 하나씩 포함하므로, 2-2 배정 일관성 점검에서 이어서 확인한다.

### 2-2. 배정 일관성 (불일치·중복 데이터 지워보기)

| 단계 | 확인 내용 | 결과 |
|---|---|---|
| a | 그룹 × 페이지 조합별 개수 | 어긋난 조합 2개 발견 |
| b | 어긋난 행 개수 | 3,893행 |
| c | 어긋난 행 제거 | 294,478 → 290,585행 |
| d | 남은 중복 제거 | 290,585 → 290,584행 |

In [9]:
# a. 데이터 조합별 개수 확인하기
pd.crosstab(df['group'], df['landing_page'])

landing_page,new_page,old_page
group,,
control,1928,145274
treatment,145311,1965


In [10]:
# b. 어긋난 행 개수 확인하기
mismatch = ((df['group'] == 'control')   & (df['landing_page'] == 'new_page')) | \
           ((df['group'] == 'treatment') & (df['landing_page'] == 'old_page'))

df['mismatch'] = mismatch      # 2-3에서 다시 쓰려고 컬럼으로 저장
mismatch.sum()                 # 3893

np.int64(3893)

In [11]:
# c. 행 제거해보기
df_clean = df[~df['mismatch']].copy()   # 원본 df는 2-3에서 쓰니까 그대로 둠
len(df_clean)                           # 290585

290585

In [12]:
# d. 남은 중복 확인하고 제거하기
print('남은 중복:', df_clean['user_id'].duplicated().sum())        # 1
display(df_clean[df_clean['user_id'].duplicated(keep=False)])      # user 773192

df_clean = df_clean.drop_duplicates('user_id', keep='first')
print('최종 행 수:', len(df_clean))                                 # 290584

남은 중복: 1


,user_id,timestamp,group,landing_page,converted,mismatch
1899,773192,2017-01-09 05:37:58.781806,treatment,new_page,0,False
2893,773192,2017-01-14 02:55:59.590927,treatment,new_page,0,False


최종 행 수: 290584


### 2-3. 배정 일관성 (지우기 전에 다시 생각해보기)

#### a. 가설 세워보기

**가설: "new_page를 보고 나서 일부러 old_page를 찾아갔다"**

이 가설이 맞다면 데이터에 다음 흔적이 있어야 한다.

| 번호 | 확인 항목 | 가설이 맞다면 |
|---|---|---|
| ① | 중복 유형 | 모두 "같은 그룹에서 두 페이지를 본" 사람이어야 한다 |
| ② | 순서 | new_page를 **먼저** 보고 old_page를 **나중에** 봐야 한다 |
| ③ | 방향 | treatment에서만 일어나야 한다 (control은 new_page의 존재를 모름) |
| ④ | 전환율 | 특별한 이유가 있는 사람들이니 일반 사용자와 달라야 한다 |

#### b. 안 지운 데이터 확인하기

| 번호 | 확인 항목 | 확인 내용 |
|---|---|---|
| ① | 중복 유형 | 2-1 d에서 확인한 결과 다시 보기 |
| ② | 순서 | 어긋난 페이지를 먼저 봤는가? |
| ③ | 방향 | treatment에서만 일어났는가? |
| ④ | 전환율 | 일반 사용자와 다른가? |

#### b. 안 지운 데이터 확인하기

In [13]:
#① 중복 유형 (2-1 d에서 확인한 결과 다시 보기)
dup_type['유형'].value_counts()
# 그룹 같음 · 페이지 다름   1998
# 페이지 같음 · 그룹 다름   1895  ← 같은 페이지를 봤으니 "찾아갔다"로 설명할 수 없음
# 완전히 같은 기록             1

유형
그룹 같음 · 페이지 다름    1998
페이지 같음 · 그룹 다름    1895
완전히 같은 기록            1
Name: count, dtype: int64

In [14]:
#② 순서: 어긋난 페이지를 먼저 봤는가?
df['timestamp'] = pd.to_datetime(df['timestamp'])

# mismatch 컬럼이 포함된 상태로 중복 사용자를 다시 뽑기
dup = df[df['user_id'].duplicated(keep=False)].sort_values(['user_id', 'timestamp'])

# 사용자별 첫 방문 기록이 어긋난 행이었던 비율
dup.groupby('user_id').first()['mismatch'].mean()     # 약 0.50

np.float64(0.5005136106831022)

In [15]:
#③ 방향: treatment에서만 일어났는가?
df[df['mismatch']]['group'].value_counts()
# treatment  1965
# control    1928

group
treatment    1965
control      1928
Name: count, dtype: int64

In [16]:
#④ 전환율: 일반 사용자와 다른가?
normal_users = df[~df['user_id'].isin(dup['user_id'])]

print('어긋난 기록 전환율:', round(dup[dup['mismatch']]['converted'].mean() * 100, 1), '%')   # 12.4
print('중복자 정상 기록 :', round(dup[~dup['mismatch']]['converted'].mean() * 100, 1), '%')  # 13.0
print('일반 사용자     :', round(normal_users['converted'].mean() * 100, 1), '%')           # 11.9

어긋난 기록 전환율: 12.4 %
중복자 정상 기록 : 13.0 %
일반 사용자     : 11.9 %


#### 📝 2-3 정리
| 확인 항목 | 가설이 맞다면 | 실제 결과 | 판정 |
|---|---|---|---|
| ① 중복 유형 | 모두 두 페이지를 봄 | 절반(1,895명)은 같은 페이지·그룹 라벨만 바뀜 | ❌ |
| ② 순서 | new → old | 반반 (50%) | ❌ |
| ③ 방향 | treatment에서만 | 양쪽 대칭 (1,965 vs 1,928) | ❌ |
| ④ 전환율 | 일반 사용자와 다름 | 비슷함 (12.4% / 13.0% / 11.9%) | ❌ |

→ 가설을 뒷받침하는 흔적이 없다. **사용자의 의도적 행동보다 시스템 오류(노출·기록·라벨 오류)로 판단**하여 2-2의 제거를 유지한다.

### 2-4. SRM 점검

50:50으로 나눴다면 그룹 크기도 대략 반반이어야 한다.
한쪽이 유독 많으면 **배정 로직에 버그**가 있다는 신호이고, 그러면 무작위 배정이 깨져 검정 결과를 믿을 수 없음

| 구분 | 내용 |
|---|---|
| 귀무가설 H₀ | 실제 그룹 비율 = 의도한 비율 (50:50) |
| 대립가설 H₁ | 실제 그룹 비율 ≠ 의도한 비율 |
| 판단 기준 | **p가 커야(≥ 0.05) 정상** |

⚠️ 평소 검정과 해석 방향이 반대다. 여기서는 H₀가 기각되지 **않아야** 좋은 것이다.

In [17]:
# a. 그룹별 인원 확인하기
n = df_clean['group'].value_counts()
print(n)
print((n / n.sum() * 100).round(2))   # 비율(%)

group
treatment    145310
control      145274
Name: count, dtype: int64
group
treatment    50.01
control      49.99
Name: count, dtype: float64


In [18]:
# b. 카이제곱 적합도 검정하기
from scipy import stats

expected = [len(df_clean) / 2] * 2           # 50:50일 때 기대 인원
chi2_srm, p_srm = stats.chisquare(n.values, expected)
print(f'χ² = {chi2_srm:.4f}, p = {p_srm:.3f}')   # χ² = 0.0045, p = 0.947

χ² = 0.0045, p = 0.947


In [19]:
# c. 정제 전 데이터도 확인하기
n_raw = df['group'].value_counts()
chi2_raw, p_raw = stats.chisquare(n_raw.values, [len(df) / 2] * 2)
print(f'정제 전: χ² = {chi2_raw:.4f}, p = {p_raw:.3f}')   # p = 0.892

정제 전: χ² = 0.0186, p = 0.892


#### 📝 2-4 정리
| 데이터 | control | treatment | p-value | 판정 |
|---|---|---|---|---|
| 정제 전 | 147,202 | 147,276 | 0.892 | ✅ 정상 |
| 정제 후 | 145,274 | 145,310 | 0.947 | ✅ 정상 |

- 정제 전후 모두 p > 0.05 → 50:50 배정과 다르다고 볼 근거가 없다.
- 정제 과정에서 두 그룹이 비슷하게 제거되어 비율이 유지되었다.
- **무작위 배정이 의도대로 이루어졌다고 판단**하고 검정을 진행한다.

### 2-5. 민감도 분석 (지운 경우 vs 안 지운 경우)

 **지우든 안 지우든 결론이 같다면** 제거 결정이 결과를 좌우하지 않았다는 근거가 된다.

| 방식 | 데이터 | 기준 |
|---|---|---|
| 지운 경우 | df_clean (290,584명) | 정제된 데이터 |
| 안 지운 경우 | ITT (290,584명) | 실제로 본 페이지와 상관없이 **처음 배정된 그룹** 기준 |

* ITT(Intention-To-Treat, 인텐션 투 트릿): "배정된 대로 분석한다"는 원칙. 실험 설계를 그대로 존중하는 방식이다.

In [20]:
#a.안지운 데이터 만들기
df_itt = (df.sort_values('timestamp')
            .groupby('user_id')
            .agg(group=('group', 'first'),          # 첫 방문 때의 그룹
                 converted=('converted', 'max')))   # 한 번이라도 전환했으면 1

len(df_itt)   # 290584

290584

In [21]:
#b. 두 데이터의 결과 비교하기
from statsmodels.stats.proportion import proportions_ztest

def compare(data, name):
    g = data.groupby('group')['converted'].agg(['sum', 'count', 'mean'])
    z, p = proportions_ztest(g.loc[['treatment', 'control'], 'sum'],
                             g.loc[['treatment', 'control'], 'count'])
    diff = (g.loc['treatment', 'mean'] - g.loc['control', 'mean']) * 100
    return {'방식': name,
            'control 전환율(%)':   round(g.loc['control', 'mean'] * 100, 2),
            'treatment 전환율(%)': round(g.loc['treatment', 'mean'] * 100, 2),
            '차이(%p)': round(diff, 3),
            'p-value':  round(p, 3)}

pd.DataFrame([compare(df_clean, '지운 경우'),
              compare(df_itt,   '안 지운 경우(ITT)')])

,방식,control 전환율(%),treatment 전환율(%),차이(%p),p-value
0,지운 경우,12.04,11.88,-0.158,0.190
1,안 지운 경우(ITT),12.17,12.04,-0.138,0.254


#### 📝 2-5 정리
- 두 방식 모두 treatment가 control보다 약 0.15%p 낮고, 통계적으로 유의하지 않다 (p > 0.05).
- **제거 여부가 결론을 바꾸지 않으므로**, 이후 분석은 정제된 `df_clean`으로 진행한다.

## 3. 질문과 가설 정하기

> ⚠️ 여기서 정한 가설·유의수준·검정 방향은 끝까지 바꾸지 않는다.    
> 결과를 보고 가설을 바꾸면 p-해킹이 된다.

### 3-1. 알고 싶은 것 (한 문장)

**"새 랜딩 페이지로 바꾸면 기존 페이지보다 전환율이 달라질까?"**

### 3-2. 가설

| 구분 | 문장 | 수식 |
|---|---|---|
| 귀무가설 H₀ | 새 페이지와 기존 페이지의 전환율은 **같다** | p_new = p_old |
| 대립가설 H₁ | 새 페이지와 기존 페이지의 전환율은 **다르다** | p_new ≠ p_old |

### 3-3. 유의수준과 검정 방향

| 항목 | 결정 | 이유 |
|---|---|---|
| 유의수준 α | 0.05 | 일반적인 기준 |
| 검정 방향 | **양측 검정** | 새 페이지가 더 좋을 수도, **더 나쁠 수도** 있다. 나빠지는 경우도 놓치면 안 된다 | 

    
> 📝 참고: 2-4 민감도 분석에서 결과를 먼저 확인했지만, 가설과 검정 방향은   
> 결과가 아닌 위의 이유로 정했다. 단측 검정이 유리해 보이도록 바꾸지 않는다.

## 4. 검정 고르기

### 4-1. 세 가지 질문에 답하기

| 질문 | 답 | 근거 |
|---|---|---|
| ① 성공 지표가 비율인가, 연속형인가? | **비율** | converted는 0(미전환) / 1(전환) 두 값뿐 |
| ② 비교할 집단이 몇 개인가? | **2개** | control vs treatment |
| ③ 정규성·등분산은 그럴듯한가? | **해당 없음** | 연속형 지표일 때만 확인 |

### 4-2. 선택한 검정

**→ 두 표본 비율 z검정 (Two-proportion z-test)**

| 지표 | 집단 수 | 검정 |
|---|---|---|
| **비율** | **2개** | **두 표본 비율 z검정** (또는 카이제곱 검정) ← 선택 |
| 비율 | 3개 이상 | 카이제곱 검정 |
| 연속형 (정규성 O) | 2개 | t검정 |
| 연속형 (정규성 X) | 2개 | Mann-Whitney U 검정 |
| 연속형 | 3개 이상 | ANOVA / Kruskal-Wallis |

- 표본이 커서(각 그룹 약 14.5만 명) z검정의 정규 근사 조건을 충분히 만족한다.
- 2×2 표의 카이제곱 검정도 같은 결과를 준다 (χ² = z²).

In [22]:
df_clean['converted'].unique()          # array([0, 1])
df_clean['group'].value_counts()        # 집단 2개 확인

group
treatment    145310
control      145274
Name: count, dtype: int64

## 5. 검정 수행하고 3종 세트로 보고하기

| 무엇을 | 왜 |
|---|---|
| ① 점추정 (차이) | 얼마나 차이 나는가 |
| ② 95% 신뢰구간 | 그 추정이 얼마나 불확실한가 |
| ③ 효과 크기 | 그 차이가 큰 차이인가 (비율 → 절대 차이 %p + 상대 차이 둘 다) |

In [23]:
#5-1. 그룹별 전환율 확인하기
summary = df_clean.groupby('group')['converted'].agg(['sum', 'count', 'mean'])
summary.columns = ['전환 수', '사용자 수', '전환율']
summary

,전환 수,사용자 수,전환율
group,,,
control,17489,145274,0.120386
treatment,17264,145310,0.118808


In [24]:
#5-2. 검정 수행하기 (두 표본 비율 z검정)
from statsmodels.stats.proportion import proportions_ztest

x_t, n_t = summary.loc['treatment', ['전환 수', '사용자 수']]
x_c, n_c = summary.loc['control',   ['전환 수', '사용자 수']]
p_t, p_c = x_t / n_t, x_c / n_c

z, p_value = proportions_ztest([x_t, x_c], [n_t, n_c])   # 양측 검정이 기본값
print(f'z = {z:.3f}, p = {p_value:.3f}')                 # z = -1.311, p = 0.190

z = -1.311, p = 0.190


In [25]:
#5-3. ① 점추정 + ② 95% 신뢰구간
from statsmodels.stats.proportion import confint_proportions_2indep

# 절대 차이 (treatment - control)
diff = p_t - p_c
ci_low, ci_high = confint_proportions_2indep(x_t, n_t, x_c, n_c, compare='diff', method='wald')
print(f'절대 차이: {diff*100:+.2f}%p  [95% CI {ci_low*100:+.2f}, {ci_high*100:+.2f}]')

# 상대 차이 (treatment / control)
ratio = p_t / p_c
r_low, r_high = confint_proportions_2indep(x_t, n_t, x_c, n_c, compare='ratio')
print(f'상대 차이: {(ratio-1)*100:+.1f}%  [95% CI {(r_low-1)*100:+.1f}, {(r_high-1)*100:+.1f}]')

절대 차이: -0.16%p  [95% CI -0.39, +0.08]
상대 차이: -1.3%  [95% CI -3.2, +0.7]


In [26]:
#5-4. 효과 크기
from scipy import stats
from statsmodels.stats.proportion import proportion_effectsize
import numpy as np

# Cramér's V (2×2 카이제곱 기반)
table = pd.crosstab(df_clean['group'], df_clean['converted'])
chi2, p_chi, dof, _ = stats.chi2_contingency(table, correction=False)
cramers_v = np.sqrt(chi2 / len(df_clean))

# Cohen's h (두 비율 전용 효과 크기)
cohens_h = proportion_effectsize(p_t, p_c)

print(f'χ²({dof}) = {chi2:.2f}, p = {p_chi:.3f}')   # χ²(1) = 1.72, p = 0.190  ← z검정과 같음 (χ² = z²)
print(f"Cramér's V = {cramers_v:.4f}")             # 0.0024
print(f"Cohen's h  = {cohens_h:.4f}")              # -0.0049

χ²(1) = 1.72, p = 0.190
Cramér's V = 0.0024
Cohen's h  = -0.0049


### 5-5. 📝 결과 보고

| 항목 | 값 |
|---|---|
| 전환율 | control 12.04% vs treatment 11.88% |
| ① 점추정 (절대) | **−0.16%p** |
| ① 점추정 (상대) | **−1.3%** (0.987배) |
| ② 95% 신뢰구간 | 절대 −0.39%p ~ +0.08%p / 상대 −3.2% ~ +0.7% |
| 검정 결과 | z = −1.31, χ²(1) = 1.72, **p = .190** |
| ③ 효과 크기 | Cramér's V = 0.002, Cohen's h = −0.005 (아주 작음) |

**보고 문장**
> 새 페이지의 전환율은 기존 페이지보다 약간 낮았지만, 통계적으로 유의한 차이는 아니었다 (11.88% vs 12.04%).
> `−0.16%p [95% CI −0.39, +0.08], 상대 −1.3%, χ²(1) = 1.72, p = .190, Cramér's V = 0.002`

**해석**
- 신뢰구간이 0을 포함한다 → "차이가 있다"고 말할 증거가 부족하다.
- 하지만 구간의 위쪽 끝도 **+0.08%p**에 불과하다 → 새 페이지가 의미 있게 **더 좋을** 가능성은 낮다.

## 6. 한 걸음 더 — A. 이 실험은 효과를 잡을 '힘'이 있었나?


5장에서 p = 0.190으로 유의하지 않았다. 하지만 p ≥ 0.05는 "차이가 없다는 증명"이 아니라 "차이가 있다는 증거가 부족하다"는 뜻으로 **효과가 정말 없는 건지, 표본이 작아서 못 잡은 건지** 검정력으로 확인 필요

| 용어 | 뜻 |
|---|---|
| 검정력 (power) | 실제로 효과가 있을 때, 그것을 **잡아낼 확률** (보통 80%) |
| MDE (Minimum Detectable Effect) | 지금 표본으로 80% 확률로 잡을 수 있는 **가장 작은 효과** |
| Cohen's h | 두 비율의 차이를 표준화한 효과 크기 (검정력 계산에 사용) |

In [27]:
# 6-1. 준비: 기준값과 변환 함수
import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

power_analysis = NormalIndPower()

p_base = p_c          # 기준 전환율 = control 전환율 (12.04%)
alpha  = 0.05
power  = 0.8

# Cohen's h → 전환율(%p)로 되돌리는 함수
def h_to_p(h, p_base):
    return np.sin(np.arcsin(np.sqrt(p_base)) + h / 2) ** 2

In [28]:
# 6-2. +1%p를 잡으려면 그룹당 몇 명이 필요한가?
h_1pp = proportion_effectsize(p_base + 0.01, p_base)    # 12.04% → 13.04%

n_needed = power_analysis.solve_power(effect_size=h_1pp, alpha=alpha,
                                      power=power, ratio=1,
                                      alternative='two-sided')
print(f"Cohen's h = {h_1pp:.4f}")
print(f'필요한 표본: 그룹당 {np.ceil(n_needed):,.0f}명')

Cohen's h = 0.0302
필요한 표본: 그룹당 17,209명


In [29]:
# 6-3. MDE: 지금 표본으로 잡을 수 있는 가장 작은 효과는?
h_mde = power_analysis.solve_power(nobs1=n_t, alpha=alpha, power=power,
                                   ratio=n_c / n_t,
                                   alternative='two-sided')   # effect_size를 비워둠

p_mde = h_to_p(h_mde, p_base)
print(f"MDE (Cohen's h) = {h_mde:.4f}")
print(f'MDE (절대)      = +{(p_mde - p_base)*100:.2f}%p   ({p_base*100:.2f}% → {p_mde*100:.2f}%)')
print(f'MDE (상대)      = +{(p_mde / p_base - 1)*100:.1f}%')

MDE (Cohen's h) = 0.0104
MDE (절대)      = +0.34%p   (12.04% → 12.38%)
MDE (상대)      = +2.8%


In [30]:
# 6-4. 효과 크기별 검정력 표
rows = []
for d in [0.001, 0.002, 0.0034, 0.005, 0.01]:
    h = proportion_effectsize(p_base + d, p_base)
    pw = power_analysis.power(effect_size=h, nobs1=n_t, alpha=alpha, ratio=n_c / n_t)
    n  = power_analysis.solve_power(effect_size=h, alpha=alpha, power=power, ratio=1)
    rows.append({'효과(%p)': f'+{d*100:.2f}',
                 '현재 표본의 검정력': f'{pw*100:.0f}%',
                 '80%에 필요한 그룹당 인원': f'{np.ceil(n):,.0f}'})
pd.DataFrame(rows)

,효과(%p),현재 표본의 검정력,80%에 필요한 그룹당 인원
0,+0.10,13%,"1,668,235"
1,+0.20,38%,"418,541"
2,+0.34,80%,"145,539"
3,+0.50,98%,"67,673"
4,+1.00,100%,"17,209"


### 6-5. 📝 정리

- +1%p를 80% 검정력으로 잡으려면 그룹당 **17,209명**이면 충분하다.
  우리는 그룹당 약 **14.5만 명**으로 필요한 인원의 8배 이상을 모았다.
- 지금 표본의 MDE는 **+0.34%p (상대 +2.8%)** 이다.

> **결론: "우리 표본은 0.34%p 이상의 효과라면 80% 확률로 잡을 수 있었다."**

- 따라서 이번 결과는 **"힘이 없어서 못 잡은 것"이 아니다.**
  새 페이지에 0.34%p 이상의 개선 효과가 있었다면 잡혔을 가능성이 높다.
- 5장의 신뢰구간 위쪽 끝(+0.08%p)도 MDE보다 작다.
  → 새 페이지가 **의미 있게 더 좋을 가능성은 낮다**고 말할 수 있다.
- 단, 0.1~0.2%p 수준의 아주 작은 효과는 이 표본으로 잡기 어렵다 (검정력 13~38%).
  이런 작은 효과까지 확인하려면 그룹당 수십만~백만 명 이상이 필요하다.

## 7. 결론 및 의사결정

### 7-1. 질문과 가설 — 무엇을 알고 싶었나

**"새 랜딩 페이지로 바꾸면 기존 페이지보다 전환율이 달라질까?"**

| 구분 | 내용 |
|---|---|
| H₀ | 새 페이지와 기존 페이지의 전환율은 같다 (p_new = p_old) |
| H₁ | 새 페이지와 기존 페이지의 전환율은 다르다 (p_new ≠ p_old) |
| 유의수준 / 방향 | α = 0.05, 양측 검정 (새 페이지가 더 나쁠 가능성도 확인) |

---

### 7-2. 데이터 점검 — 무엇을 어떻게 정제했나

| 단계 | 행 수 | 제거 | 이유 |
|---|---|---|---|
| 원본 | 294,478 | – | 결측치 없음 |
| 배정 불일치 제거 | 290,585 | 3,893 | 그룹과 본 페이지가 어긋나 실제 노출 페이지를 확정할 수 없음 |
| user_id 중복 제거 | 290,584 | 1 | 같은 기록이 2번 저장됨 (1인 1관측 원칙) |

- 지운 사용자가 의도적으로 다른 페이지를 찾아간 것인지 검토함 → 순서·방향·전환율 모두 패턴이 없어 **시스템 오류로 판단**
- SRM: 정제 전 p = 0.892, 정제 후 p = 0.947 → **50:50 배정 정상**
- 민감도 분석: 지운 경우와 안 지운 경우(ITT) 모두 차이 약 −0.15%p, p > 0.05 → **제거 여부가 결론을 바꾸지 않음**

---

### 7-3. 검정 선택 근거 — 왜 그 검정인가

성공 지표인 전환 여부(converted)는 0과 1뿐인 **비율 지표**이고, 비교 집단은 control과 treatment **2개**다.
따라서 **두 표본 비율 z검정**을 선택했다. 연속형 지표가 아니므로 정규성·등분산 가정은 해당하지 않으며,
각 그룹 약 14.5만 명, 전환 수 약 1.7만 건으로 정규 근사 조건(np ≥ 10, n(1−p) ≥ 10)을 충분히 만족한다.
2×2 카이제곱 검정으로도 같은 결과를 확인했다 (χ² = z²).

---

### 7-4. 결과 — 점추정 · 신뢰구간 · 효과 크기

| 항목 | 값 |
|---|---|
| 전환율 | control **12.04%** vs treatment **11.88%** |
| 점추정 (절대) | **−0.16%p** |
| 점추정 (상대) | **−1.3%** |
| 95% 신뢰구간 | 절대 −0.39%p ~ +0.08%p / 상대 −3.2% ~ +0.7% |
| 검정 | z = −1.31, χ²(1) = 1.72, **p = .190** |
| 효과 크기 | Cramér's V = 0.002, Cohen's h = −0.005 (아주 작음) |
| 검정력 (MDE) | 현재 표본으로 **+0.34%p 이상**의 효과는 80% 확률로 잡을 수 있음 |

> 새 페이지의 전환율은 기존 페이지보다 약간 낮았지만, 통계적으로 유의한 차이는 아니었다.
> `−0.16%p [95% CI −0.39, +0.08], 상대 −1.3%, χ²(1) = 1.72, p = .190, Cramér's V = 0.002`

---

### 7-5. 의사결정

## ❌ 출시하지 않는다 (기존 페이지 유지)

| 선택지 | 판단 | 근거 |
|---|---|---|
| 출시 | ❌ | 새 페이지가 더 좋다는 증거가 없다. 점추정은 오히려 −0.16%p로 낮다 |
| 데이터 추가 수집 | ❌ | 이미 표본이 충분하다 (+1%p 검출에 필요한 인원의 8배 이상). 신뢰구간 위쪽 끝도 +0.08%p라, 더 모아도 잡을 수 있는 것은 실무적으로 의미 없는 아주 작은 효과뿐이다 |
| **유지 (출시 안 함)** | ✅ | 아래 근거 참고 |

**근거**
1. **유의하지 않다:** p = 0.190으로 차이가 있다는 증거가 부족하다.
2. **힘이 부족해서가 아니다:** MDE가 +0.34%p이므로, 새 페이지에 그 이상의 개선 효과가 있었다면 잡혔을 가능성이 높다.
3. **좋아질 여지도 작다:** 신뢰구간으로 보면 최선의 경우에도 +0.08%p 개선에 그친다. 페이지 교체에 드는 개발·디자인·운영 비용을 감안하면 도입할 이유가 없다.
4. **결과가 견고하다:** 데이터를 지우든 안 지우든 결론이 같다.

> **"새 랜딩 페이지는 전환율을 개선하지 못했다. 기존 페이지를 유지하고, 다른 개선안을 실험하는 데 자원을 쓰는 것이 낫다."**

---

### 7-6. 한계 — 이 결론이 기대는 가정, 이 데이터로 알 수 없는 것

**이 결론이 기대는 가정**
- 배정이 무작위로 이루어졌다 (SRM 점검으로 일부 확인했지만, 배정 로직 자체는 확인할 수 없다)
- 제거한 3,893행은 무작위 시스템 오류다 (원인이 노출 오류·기록 오류·라벨 오류 중 무엇인지는 구분할 수 없다)
- 사용자 간 서로 영향을 주지 않는다 (한 사람의 행동이 다른 사람의 전환에 영향을 주지 않음)

**이 데이터로 알 수 없는 것**
- **장기 효과:** 실험 기간이 약 3주(2017-01-02 ~ 01-24)뿐이다. 새 페이지에 익숙해지는 데 시간이 걸리는 효과(변화 거부감)나, 처음에만 반짝 관심을 끄는 효과(신기 효과)는 확인할 수 없다.
- **다른 지표:** 전환 여부만 있다. 구매 금액, 재방문, 체류 시간 등에서 새 페이지가 나을 가능성은 이 데이터로 판단할 수 없다.
- **아주 작은 효과:** 0.1~0.2%p 수준의 효과는 검정력이 13~38%에 불과해 있는지 없는지 판단할 수 없다.
- **세부 집단별 차이:** 특정 사용자 층(신규/기존, 기기 종류 등)에서만 효과가 있을 가능성은 확인하지 않았다.
- **시점:** 2017년 데이터라 지금의 사용자 행동과 다를 수 있다.

**분석 과정의 한계**
- 민감도 분석에서 검정 결과를 가설 설정 전에 먼저 확인했다. 가설과 검정 방향은 결과와 무관한 이유(양측)로 정했지만, 엄밀한 사전 등록은 아니다.
- 실무적으로 의미 있는 최소 효과(예: +0.5%p)를 실험 전에 정해두지 않았다.